# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

METHOD: K-Means Clustering

WHY K-Means?
- Interpretable: cluster centers show archetype profile directly
- Scalable: ~182K items in seconds
- Silhouette score is standard metric for cluster quality
- Matches w02 framing: "discover natural performance groupings"

WHY NOT alternatives?
- DBSCAN: unclear density thresholds; risk of "noise" cluster dominating
- Hierarchical: expensive on 182K items; dendrograms hard to act on
- Gaussian Mixture: more complexity, no semantic gain over K-Means

FEATURES for clustering (will be standardized):
1. ctr — click-through rate (engagement signal: clicks ÷ impressions)
2. avg_position — average search rank (visibility, ranking quality)
3. impressions_log — log10(impressions + 1) (compress high-volume tail)

These three dimensions capture:
  - How well the page appeals to searchers (CTR)
  - How visible it is (position)
  - How much reach it has (volume)

EXPECTED ARCHETYPES (emergent, discovered by clustering):
1. CHAMPION: top-5 position + high CTR + high volume → protect/expand
2. RISING_STAR: improving position + moderate CTR + growing volume → optimize
3. TITLE_CTR_PROBLEM: good position + low CTR + high volume → fix meta
4. NICHE_OPPORTUNITY: moderate CTR + low volume + mid position → test/expand
5. DEAD_WEIGHT: poor position + low CTR + low volume → consolidate/delete

VALIDATION:
- Silhouette score (cluster cohesion/separation)
- Does cluster ID correlate with baseline score? (ANOVA)
- → YES: clusters segment items the baseline also recognizes as different priority
- → NO: clustering uses different dimensions than baseline (still valid, different lens)

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Why grouped split (by client)?

Content items within a single client are more similar to each other than random items across clients. If we split randomly by rows, K-Means will fit clusters that may just memorize client-specific behavior. A grouped split by client_hash_id tests: "Does the cluster structure from one client's content apply to another client?"

This prevents data leakage in the sense that we're not letting one client's behavior dominate the cluster centroids.

In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import matplotlib.pyplot as plt
import os

os.makedirs('work/outputs', exist_ok=True)

df = pd.read_csv('work/outputs/baseline_action_score.csv')

print(f"Loaded {len(df):,} scored items from baseline")
print(f"Columns: {df.columns.tolist()}")
print(f"\nAction label breakdown:")
print(df['action_label'].value_counts())

Loaded 182,135 scored items from baseline
Columns: ['content_hash_id', 'client_hash_id', 'impressions', 'clicks', 'ctr', 'avg_position', 'expected_ctr', 'ctr_gap', 'score', 'position_bucket', 'reason_code', 'action_label', 'days_active', 'first_date', 'last_date']

Action label breakdown:
action_label
NO_ACTION_CTR_ON_TRACK          160982
FIX_TITLE_META_MONITOR           15882
FIX_TITLE_META_HIGH_PRIORITY      5271
Name: count, dtype: int64


In [12]:
# Define clustering features
features_for_clustering = ['ctr', 'avg_position']

# Add log-impressions to compress high-volume tail
df['impressions_log'] = np.log10(df['impressions'] + 1)
features_for_clustering.append('impressions_log')

print(f"\nClustering features available: {features_for_clustering}")
print(f"(Note: engagement_rate and scroll_rate not in warehouse baseline output)")

print(f"\nFeature summary (before standardization):")
print(df[features_for_clustering].describe())

# Check for missing values
print(f"\nMissing values per feature:")
for col in features_for_clustering:
    pct_missing = 100 * df[col].isna().sum() / len(df)
    print(f"  {col}: {pct_missing:.2f}%")

# Fill any missing values
for col in features_for_clustering:
    df[col] = df[col].fillna(df[col].median())


Clustering features available: ['ctr', 'avg_position', 'impressions_log']
(Note: engagement_rate and scroll_rate not in warehouse baseline output)

Feature summary (before standardization):
                 ctr   avg_position  impressions_log
count  182135.000000  182135.000000    182135.000000
mean        0.003172      18.002448         3.253352
std         0.005406      14.707742         0.782179
min         0.000000       0.000000         2.004321
25%         0.000000       7.600625         2.608526
50%         0.001716      12.772799         3.180126
75%         0.004090      23.440158         3.819215
max         0.536745     173.269278         6.462790

Missing values per feature:
  ctr: 0.00%
  avg_position: 0.00%
  impressions_log: 0.00%


In [13]:
# Grouped train/test split

# Get unique clients
unique_clients = df['client_hash_id'].unique()
print(f"Total unique clients: {len(unique_clients)}")

# 80/20 split by client (grouped split)
np.random.seed(42)
np.random.shuffle(unique_clients)

split_idx = int(0.8 * len(unique_clients))
train_clients = set(unique_clients[:split_idx])
test_clients = set(unique_clients[split_idx:])

# Create train/test masks
train_mask = df['client_hash_id'].isin(train_clients)
test_mask = df['client_hash_id'].isin(test_clients)

print(f"\nTrain/test split (by client):")
print(f"  Train: {len(train_clients)} clients, {train_mask.sum():,} items")
print(f"  Test:  {len(test_clients)} clients, {test_mask.sum():,} items")

# Extract feature matrices
X_full = df[features_for_clustering].values
X_train = X_full[train_mask]
X_test = X_full[test_mask]

print(f"\nFeature matrix shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test: {X_test.shape}")

# Standardize BOTH on train mean/std (prevent test leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_full_scaled = scaler.transform(X_full)

print(f"\n Features standardized (mean=0, std=1)")
print(f"  Train scaled mean: {X_train_scaled.mean(axis=0)}")
print(f"  Train scaled std: {X_train_scaled.std(axis=0)}")

Total unique clients: 59

Train/test split (by client):
  Train: 47 clients, 161,153 items
  Test:  12 clients, 20,982 items

Feature matrix shapes:
  X_train: (161153, 3)
  X_test: (20982, 3)

 Features standardized (mean=0, std=1)
  Train scaled mean: [-4.18727351e-13 -1.85302522e-14 -5.15088892e-13]
  Train scaled std: [1. 1. 1.]


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [14]:
# Train + Compare

print("SECTION 3: K-MEANS TRAINING & CLUSTER VALIDATION")
print("-"*70)


# K-selection: Fit k=3 to k=9, report metrics

print("\nK-SELECTION (k=3 to k=9):")
print("(Fitting ~3-5 minutes)\n")

k_range = range(3, 10)
results = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300, verbose=0)

    # Fit on train
    train_labels = km.fit_predict(X_train_scaled)
    train_sil = silhouette_score(X_train_scaled, train_labels)
    inertia = km.inertia_

    # Predict on test
    test_labels = km.predict(X_test_scaled)
    test_sil = silhouette_score(X_test_scaled, test_labels)

    results.append({
        'k': k,
        'inertia': inertia,
        'train_sil': train_sil,
        'test_sil': test_sil,
        'km': km
    })

    print(f"k={k}: inertia={inertia:,.0f} | train_sil={train_sil:.3f} | test_sil={test_sil:.3f}")

results_df = pd.DataFrame(results)

print("\n K-selection complete")


best_k_idx = results_df['test_sil'].idxmax()
k_best = results_df.loc[best_k_idx, 'k']
best_test_sil = results_df.loc[best_k_idx, 'test_sil']

print(f"\nCHOSEN k = {int(k_best)}")
print(f"  (Highest test silhouette: {best_test_sil:.3f})")
print(f"  Reason: Best generalization to held-out clients")


print(f"\nFitting K-Means with k={int(k_best)} on all {len(df):,} items...")

km_final = KMeans(n_clusters=int(k_best), random_state=42, n_init=10, max_iter=300)
cluster_labels = km_final.fit_predict(X_full_scaled)

df['cluster'] = cluster_labels

print(f" Clustering complete\n")


print("CLUSTER SIZE DISTRIBUTION:")
print(f"{'─'*50}\n")

cluster_sizes = pd.Series(cluster_labels).value_counts().sort_index()
size_table = pd.DataFrame({
    'cluster': cluster_sizes.index,
    'n_items': cluster_sizes.values,
    'pct': (100 * cluster_sizes.values / len(df)).round(1)
})

print(size_table.to_string(index=False))



# Silhouette score table

sample_silhouette_values = silhouette_samples(X_full_scaled, cluster_labels)
df['silhouette'] = sample_silhouette_values

print(f"\n\nSILHOUETTE SCORE STATISTICS:")
print(f"{'─'*50}\n")

sil_stats = pd.DataFrame({
    'Metric': ['Mean', 'Std', 'Min', 'Max', '% items (sil > 0)'],
    'Value': [
        f"{sample_silhouette_values.mean():.3f}",
        f"{sample_silhouette_values.std():.3f}",
        f"{sample_silhouette_values.min():.3f}",
        f"{sample_silhouette_values.max():.3f}",
        f"{100 * (sample_silhouette_values > 0).sum() / len(df):.1f}%"
    ]
})

print(sil_stats.to_string(index=False))

print(f"\n\nInterpretation:")
print(f"  Mean silhouette {sample_silhouette_values.mean():.3f} indicates clusters are")
if sample_silhouette_values.mean() > 0.4:
    print(f"  well-separated and internally cohesive.")
elif sample_silhouette_values.mean() > 0.25:
    print(f"  reasonably separated (acceptable cluster quality).")
else:
    print(f"  weakly separated (clusters may be overlapping).")

SECTION 3: K-MEANS TRAINING & CLUSTER VALIDATION
----------------------------------------------------------------------

K-SELECTION (k=3 to k=9):
(Fitting ~3-5 minutes)

k=3: inertia=256,724 | train_sil=0.356 | test_sil=0.269
k=4: inertia=192,702 | train_sil=0.378 | test_sil=0.336
k=5: inertia=164,593 | train_sil=0.379 | test_sil=0.336
k=6: inertia=143,587 | train_sil=0.315 | test_sil=0.267
k=7: inertia=126,745 | train_sil=0.316 | test_sil=0.267
k=8: inertia=110,480 | train_sil=0.310 | test_sil=0.264
k=9: inertia=100,083 | train_sil=0.313 | test_sil=0.275

 K-selection complete

CHOSEN k = 4
  (Highest test silhouette: 0.336)
  Reason: Best generalization to held-out clients

Fitting K-Means with k=4 on all 182,135 items...
 Clustering complete

CLUSTER SIZE DISTRIBUTION:
──────────────────────────────────────────────────

 cluster  n_items  pct
       0    77819 42.7
       1    29177 16.0
       2    67734 37.2
       3     7405  4.1


SILHOUETTE SCORE STATISTICS:
──────────────────

In [18]:
# Model-vs-Baseline Comparison Table

from scipy.stats import f_oneway

"""
LANE 3: BASELINE COMPARISON (required by assignment)

The baseline (w04) ranks items by: score = CTR_gap × impressions

Clustering uses: CTR, position, volume interactions

Question: Does clustering add segmentation value?
Evidence: ANOVA on baseline scores by cluster
"""

print("="*70)
print("SECTION 3.5: MODEL-VS-BASELINE COMPARISON")
print("="*70)


# Model-vs-Baseline Table 1: Baseline score distribution by cluster

print(f"\nTABLE 1: Baseline Score Statistics by Cluster")
print(f"{'─'*70}\n")

cluster_score_stats = df.groupby('cluster')['score'].agg([
    ('n_items', 'count'),
    ('mean_score', 'mean'),
    ('median_score', 'median'),
    ('std_score', 'std'),
    ('min_score', 'min'),
    ('max_score', 'max'),
]).round(1)

print(cluster_score_stats.to_string())


# ANOVA: Are baseline scores significantly different by cluster?

print(f"\n{'─'*70}")
print("ANOVA Test: Baseline Score by Cluster")
print(f"{'─'*70}\n")

cluster_groups = [df[df['cluster'] == c]['score'].values for c in range(int(k_best))]
f_stat, p_val = f_oneway(*cluster_groups)

print(f"F-statistic: {f_stat:.2f}")
print(f"p-value: {p_val:.2e}")

if p_val < 0.05:
    print(f"\n SIGNIFICANT (p < 0.05)")
    print(f"Interpretation: Clusters separate baseline priorities.")
    print(f"Clustering adds segmentation value.")
else:
    print(f"\n NOT SIGNIFICANT (p >= 0.05)")
    print(f"Interpretation: Clusters don't separate baseline scores.")
    print(f"Clustering uses different dimensions than baseline.")



# Model-vs-Baseline Table 2: Action label distribution

print(f"\n{'─'*70}")
print("TABLE 2: Action Label Distribution by Cluster (w04 baseline)")
print(f"{'─'*70}\n")

action_by_cluster = pd.crosstab(
    df['cluster'],
    df['action_label'],
    margins=False
)

print("Counts:")
print(action_by_cluster.to_string())

# As percentages
print("\n\nAs % within each cluster:")
action_pct = pd.crosstab(
    df['cluster'],
    df['action_label'],
    normalize='index'
) * 100

print(action_pct.round(1).to_string())


# Summary: Model-vs-Baseline

print(f"\n{'═'*70}")
print("MODEL-vs-BASELINE SUMMARY")
print(f"{'═'*70}\n")

print("BASELINE (w04):")
print("  - Metric: score = (expected_ctr - actual_ctr) × impressions")
print("  - Logic: Single-dimensional ranking by CTR improvement opportunity")
print("  - Clusters by: HIGH_PRIORITY (score ≥100), MONITOR (20-99), NO_ACTION (<20)")

print("\nCLUSTERING (w05):")
print(f"  - Method: K-Means with k={int(k_best)} clusters")
print("  - Logic: Multi-dimensional segmentation (CTR, position, volume)")
print("  - Silhouette score (mean): {:.3f}".format(df['silhouette'].mean()))

print("\nCOMPARISON RESULT:")
if p_val < 0.05:
    print(f" - Clusters ALIGN with baseline (ANOVA p={p_val:.2e})")
    print("    Use clustering to GROUP items, then rank within groups by baseline")
else:
    print(f" - Clusters DIVERGE from baseline (ANOVA p={p_val:.2e})")
    print("    Use clustering as ALTERNATIVE segmentation (different strategic lens)")

SECTION 3.5: MODEL-VS-BASELINE COMPARISON

TABLE 1: Baseline Score Statistics by Cluster
──────────────────────────────────────────────────────────────────────

         n_items  mean_score  median_score  std_score  min_score  max_score
cluster                                                                    
0          77819         1.3           0.6        1.7        0.0       13.1
1          29177         1.6           0.5        4.1        0.0      282.8
2          67734        33.0           8.5       94.8        0.0     4176.1
3           7405         0.0           0.0        0.0        0.0        0.0

──────────────────────────────────────────────────────────────────────
ANOVA Test: Baseline Score by Cluster
──────────────────────────────────────────────────────────────────────

F-statistic: 4255.02
p-value: 0.00e+00

 SIGNIFICANT (p < 0.05)
Interpretation: Clusters separate baseline priorities.
Clustering adds segmentation value.

─────────────────────────────────────────────

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [21]:
"""
LANE 3: Unsupervised, so "errors" means:
  - Items on cluster boundaries (ambiguous archetype assignment)
  - Clusters that don't map to business archetypes (hard to interpret)
  - Outliers within clusters (items that break archetype profile)
"""

print("SECTION 4: CLUSTER PROFILES & INTERPRETATION")
print("-"*70)


# STEP 1: Cluster profiles (what does each archetype look like?)

k_best_int = int(k_best)

for c in range(k_best_int):
    cluster_data = df[df['cluster'] == c]

    print(f"\n{'='*70}")
    print(f"CLUSTER {c} (n = {len(cluster_data):,} items, {100*len(cluster_data)/len(df):.1f}%)")
    print(f"{'='*70}")

    # Performance metrics
    mean_ctr = cluster_data['ctr'].mean()
    mean_pos = cluster_data['avg_position'].mean()
    mean_imp = cluster_data['impressions'].mean()
    mean_sil = cluster_data['silhouette'].mean()

    print(f"\nPerformance footprint:")
    print(f"  CTR:                {mean_ctr:.4f} ({mean_ctr*100:.2f}%)")
    print(f"  Avg position:       {mean_pos:.1f}")
    print(f"  Mean impressions:   {mean_imp:,.0f}")
    print(f"  Median impressions: {cluster_data['impressions'].median():,.0f}")

    print(f"\nBaseline score (w04):")
    print(f"  Mean score:         {cluster_data['score'].mean():.1f}")
    print(f"  Median score:       {cluster_data['score'].median():.1f}")

    high_priority_pct = 100 * (cluster_data['action_label'] == 'FIX_TITLE_META_HIGH_PRIORITY').sum() / len(cluster_data)
    monitor_pct = 100 * (cluster_data['action_label'] == 'FIX_TITLE_META_MONITOR').sum() / len(cluster_data)
    no_action_pct = 100 * (cluster_data['action_label'] == 'NO_ACTION_CTR_ON_TRACK').sum() / len(cluster_data)

    print(f"  % HIGH_PRIORITY:    {high_priority_pct:.1f}%")
    print(f"  % MONITOR:          {monitor_pct:.1f}%")
    print(f"  % NO_ACTION:        {no_action_pct:.1f}%")

    print(f"\nCluster quality:")
    print(f"  Mean silhouette:    {mean_sil:.3f}")
    print(f"  % items (sil > 0):  {100 * (cluster_data['silhouette'] > 0).sum() / len(cluster_data):.1f}%")

    # Assign archetype
    if mean_pos < 6 and mean_ctr > 0.004:
        archetype = "CHAMPION"
        desc = "Top-ranked pages with high CTR and engagement. Protect and amplify."
    elif mean_pos < 10 and high_priority_pct > 3:
        archetype = "TITLE/CTR_PROBLEM"
        desc = "Good rankings but low CTR. Fix title/meta tags to capture clicks."
    elif mean_imp > 600:
        archetype = "HIGH_VOLUME_MIXED"
        desc = "High reach pages with variable CTR. Segment by position further."
    elif mean_imp < 250 and mean_pos < 15:
        archetype = "NICHE_OPPORTUNITY"
        desc = "Low volume but decent ranking. Test/expand with internal linking."
    else:
        archetype = "HETEROGENEOUS"
        desc = "Mixed characteristics. Review manually for sub-archetypes."

    print(f"\n  -> Archetype: {archetype}")
    print(f"    {desc}")


# STEP 2: Boundary cases (ambiguous assignments)

print("SECTION 4.2: BOUNDARY CASES (Silhouette < 0)")
print(f"{'-'*70}")

boundary_items = df[df['silhouette'] < 0]
print(f"\nItems with negative silhouette (between clusters):")
print(f"  Count: {len(boundary_items):,} ({100*len(boundary_items)/len(df):.1f}%)")
print(f"  Interpretation: These items are ambiguous — equally close to 2+ clusters")
print(f"  Strategy: Review manually if high baseline score; may reveal sub-archetypes")

# Show top high-score boundary items
high_score_boundary = boundary_items[boundary_items['score'] >= 100].nlargest(10, 'score')
if len(high_score_boundary) > 0:
    print(f"\n  High-score boundary items (score >= 100):")
    print(high_score_boundary[['content_hash_id', 'cluster', 'silhouette', 'score',
                               'ctr', 'avg_position', 'impressions']].to_string(index=False))
else:
    print(f"\n  (No high-score items on cluster boundaries)")


# STEP 3: Outliers within clusters

print("SECTION 4.3: OUTLIERS WITHIN CLUSTERS")
print(f"{'-'*70}\n")

for c in range(k_best_int):
    cluster_data = df[df['cluster'] == c]

    # Find outliers: high silhouette (well-assigned) but extreme score
    extreme_high = cluster_data[cluster_data['silhouette'] > 0.5].nlargest(1, 'score')
    extreme_low = cluster_data[cluster_data['silhouette'] > 0.5].nsmallest(1, 'score')

    if len(extreme_high) > 0:
        row = extreme_high.iloc[0]
        print(f"Cluster {c} — Highest within-cluster score:")
        print(f"  {row['content_hash_id']}: score={row['score']:.0f}, " +
              f"ctr={row['ctr']:.4f}, pos={row['avg_position']:.1f}, " +
              f"imp={row['impressions']:.0f}")


# STEP 4: What clustering adds (vs baseline)

print("SECTION 4.4: WHAT CLUSTERING ADDS (vs w04 baseline)")
print(f"{'-'*70}\n")

print("Baseline (w04): ONE-DIMENSIONAL ranking")
print("  Metric: score = (expected_ctr - actual_ctr) × impressions")
print("  Logic: 'Which items have the biggest CTR improvement opportunity?'")
print("  Blind to: ranking dynamics, engagement patterns, volume tiers\n")

print("Clustering: MULTI-DIMENSIONAL segmentation")
print("  Dimensions: CTR (appeal), position (visibility), volume (reach)")
print("  Logic: 'Which items behave alike? What archetype are they?'")
print("  Enables: tier-specific strategies (e.g., 'top-ranked stuff' vs 'emerging')\n")

print("Complementarity:")
print("  Use baseline WITHIN each cluster to rank by urgency")
print("  Use clusters ACROSS baseline to find patterns (e.g., 'all Title Problems are low-pos')")
print("  Combine: 'Fix Title/CTR Problems in high-volume cluster first'")

SECTION 4: CLUSTER PROFILES & INTERPRETATION
----------------------------------------------------------------------

CLUSTER 0 (n = 77,819 items, 42.7%)

Performance footprint:
  CTR:                0.0023 (0.23%)
  Avg position:       13.7
  Mean impressions:   807
  Median impressions: 569

Baseline score (w04):
  Mean score:         1.3
  Median score:       0.6
  % HIGH_PRIORITY:    0.0%
  % MONITOR:          0.0%
  % NO_ACTION:        100.0%

Cluster quality:
  Mean silhouette:    0.375
  % items (sil > 0):  99.9%

  -> Archetype: HIGH_VOLUME_MIXED
    High reach pages with variable CTR. Segment by position further.

CLUSTER 1 (n = 29,177 items, 16.0%)

Performance footprint:
  CTR:                0.0009 (0.09%)
  Avg position:       45.8
  Mean impressions:   1,452
  Median impressions: 569

Baseline score (w04):
  Mean score:         1.6
  Median score:       0.5
  % HIGH_PRIORITY:    0.0%
  % MONITOR:          0.7%
  % NO_ACTION:        99.3%

Cluster quality:
  Mean silhouette

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.